# PySpark Data Cleaning & Analysis — `retail_transactions_dirty.csv`

This notebook adapts the standard 15-question PySpark cleaning/analysis exercise to the **actual uploaded file**, `retail_transactions_dirty.csv` (62,425 rows), instead of a synthetic placeholder dataset.

**Actual columns in this file:**

`transaction_id, customer_id, customer_name, age, gender, category, region, quantity, unit_price, purchase_amount, rating, payment_method, purchase_date, signup_date, email`

This file is genuinely messy in ways the original synthetic dataset wasn't, which makes several of the questions much more realistic:

| Issue | Example |
|---|---|
| Inconsistent casing/whitespace in `region` | `'west'`, `'West'`, `' West '`, `'WEST'`, `'West  '` |
| Inconsistent casing/whitespace in `category` | `'furniture'`, `'Furniture'`, `' Furniture '`, `'FURNITURE'` |
| **Three different date formats mixed in `purchase_date`** | `2023-12-22` (ISO), `04/10/2022` (D/M/Y), `08-28-2021` (M-D-Y) |
| Nulls scattered across almost every column | `age`, `category`, `region`, `purchase_amount`, `payment_method`, `email`, etc. |
| Exact duplicate rows | 2,400 fully-duplicated rows |
| One fully-blank "sentinel" row | `transaction_id = 'TXN_BLANK21'` with every other field `NULL` |

Column mapping used below (synthetic → real):

| Original exercise column | Used here instead |
|---|---|
| `user_id` | `customer_id` |
| `transaction_date` | `purchase_date` |
| `product_category` | `category` |
| `sale_amount` | `purchase_amount` |
| `status` | `payment_method` |
| `city` | `payment_method` (count question) |
| `subscription` | `gender` |
| `raw_timestamp` | `purchase_date` (cast attempt) |
| `username` | `customer_name` |
| `price` | `unit_price` |
| `store_id` | `region` (cleaned) |


In [1]:
!pip install pyspark

---

## Setup & Load

Load the real CSV with `inferSchema=True` and preview it.

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, avg, sum as _sum, min as _min, max as _max, mean,
    trim, initcap, try_to_timestamp, lit
)
from pyspark.sql.types import TimestampType
import os

# Check if file exists; if not, provide a prompt to upload
file_path = "retail_transactions_dirty.csv"
if not os.path.exists(file_path):
    print(f"ERROR: {file_path} not found. Please upload the file to /content/.")
    from google.colab import files
    uploaded = files.upload()

spark = SparkSession.builder.appName("RetailCleaningTask").getOrCreate()

df = spark.read.csv(file_path, header=True, inferSchema=True)
df.printSchema()
df.show(5)

ERROR: retail_transactions_dirty.csv not found. Please upload the file to /content/.


Saving retail_transactions_dirty.csv to retail_transactions_dirty.csv
root
 |-- transaction_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- age: double (nullable = true)
 |-- gender: string (nullable = true)
 |-- category: string (nullable = true)
 |-- region: string (nullable = true)
 |-- quantity: double (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- purchase_amount: double (nullable = true)
 |-- rating: double (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- purchase_date: string (nullable = true)
 |-- signup_date: string (nullable = true)
 |-- email: string (nullable = true)

+--------------+-----------+-------------+----+------+---------+-------+--------+----------+---------------+------+----------------+-------------+-----------+--------------------+
|transaction_id|customer_id|customer_name| age|gender| category| region|quantity|unit_price|purchase_amount|rating|  paym

---

### Q1: What are the key limitations of traditional MapReduce that make Spark a preferred choice for modern big data processing?

**Disk I/O Bottleneck:** Traditional MapReduce writes intermediate processing results to physical disk (HDFS) between every single map and reduce phase.

**Latency:** The constant reading and writing to disk creates massive latency. Spark overcomes this by utilizing in-memory processing, making it substantially faster for complex, multi-step data pipelines like the one below.

### Q2: Explain how Spark uses In-Memory Computing to speed up iterative machine learning algorithms compared to disk-based systems.

Machine learning algorithms require passing over the same dataset multiple times (iterative processing) to optimize parameters.

Disk-based systems must reload the data from disk for every iteration. Spark caches the datasets (DataFrames/RDDs) directly in RAM, so subsequent iterations access this data at memory speed, drastically reducing overall execution time.

### Q3: Write a code snippet to remove all duplicate rows from a DataFrame based on a specific set of columns: `customer_id` and `purchase_date`.

(Mapped from `user_id`/`transaction_date` → `customer_id`/`purchase_date`.) This file actually contains real duplicates — 2,400 fully duplicated rows, plus a fully-blank sentinel row (`TXN_BLANK21`).

In [4]:
df_cleaned = df.dropDuplicates(["customer_id", "purchase_date"])

print(f"Original count: {df.count()}")
print(f"Cleaned count: {df_cleaned.count()}")
df_cleaned.show(5)

Original count: 62425
Cleaned count: 59862
+--------------+-----------+-------------+----+------+--------+-------+--------+----------+---------------+------+----------------+-------------+-----------+--------------------+
|transaction_id|customer_id|customer_name| age|gender|category| region|quantity|unit_price|purchase_amount|rating|  payment_method|purchase_date|signup_date|               email|
+--------------+-----------+-------------+----+------+--------+-------+--------+----------+---------------+------+----------------+-------------+-----------+--------------------+
|   TXN_BLANK21|       NULL|         NULL|NULL|  NULL|    NULL|   NULL|    NULL|      NULL|           NULL|  NULL|            NULL|         NULL|       NULL|                NULL|
|     TXN128358|  CUST10000| Ishaan Jones|43.0| Other|  Beauty|  South|     6.0|     35.99|         228.48|   4.0|          Wallet|   01-14-2022| 2022-04-25|ishaan.jones180@c...|
|     TXN100633|  CUST10000|    Ava Reddy|57.0| Other|Clothing

### Q4: Filter for rows where the region is `'West'` and group by `category` to find the average `purchase_amount`.

Because `region`/`category` contain mixed casing and stray whitespace (`'west'`, `' West '`, `'WEST'`...), a plain `== "West"` filter on the raw column would silently miss most West-region rows. We normalize both columns with `trim()` + `initcap()` first.

In [5]:
df_norm = df.withColumn("region_clean", initcap(trim(col("region")))) \
            .withColumn("category_clean", initcap(trim(col("category"))))

df_west_avg = df_norm.filter(col("region_clean") == "West") \
                      .groupBy("category_clean") \
                      .agg(avg("purchase_amount").alias("average_purchase_amount"))

df_west_avg.show()

+--------------+-----------------------+
|category_clean|average_purchase_amount|
+--------------+-----------------------+
|        Sports|     221.23952040085896|
|          NULL|     233.61123188405787|
|       Grocery|     216.85205035971222|
|   Electronics|     212.67376978417263|
|      Clothing|     215.47993124522526|
|         Books|      215.4238566308244|
|     Furniture|     212.76997938144325|
|        Beauty|      222.4497036526534|
|          Toys|     220.65586182833198|
+--------------+-----------------------+



### Q5: Difference between `.na.drop()` and `.na.fill()`. Fill nulls in `payment_method` with `'Unknown'`.

**`.na.drop()`** removes the entire row if it contains a null value.

**`.na.fill()`** replaces the null value with a specified default instead of deleting the row — preserving the rest of that row's information (mapped here from `status` → `payment_method`, since this file has no `status` column but does have 1,879 null `payment_method` values).

In [6]:
df_filled = df.na.fill({"payment_method": "Unknown"})
df_filled.select("transaction_id", "payment_method").show(5)

+--------------+----------------+
|transaction_id|  payment_method|
+--------------+----------------+
|     TXN130813|     Net Banking|
|     TXN152131|     Credit Card|
|     TXN117691|Cash on Delivery|
|     TXN157025|          Wallet|
|     TXN117326|             UPI|
+--------------+----------------+
only showing top 5 rows


### Q6: Total count of records for each value of a categorical column, keeping only groups with count > 100.

(Mapped from `city` → `payment_method`, since this file has no `city` column.)

In [7]:
df_payment_counts = df.groupBy("payment_method") \
                       .count() \
                       .filter(col("count") > 100)

df_payment_counts.show()

+----------------+-----+
|  payment_method|count|
+----------------+-----+
|     Credit Card|10147|
|            NULL| 1879|
|     Net Banking|10169|
|      Debit Card| 9962|
|          Wallet|10253|
|Cash on Delivery| 9940|
|             UPI|10075|
+----------------+-----+



### Q7: How does the immutability of Spark DataFrames affect data-cleaning steps like dropping or renaming columns?

Because DataFrames are immutable, they can't be modified in place. Every `.drop()`, `.withColumnRenamed()`, `.na.fill()`, etc. returns a **brand-new** DataFrame; the original is untouched. You must assign the result to a variable (or overwrite the existing one) for the cleaning step to "stick" — which is exactly why every cell above reassigns to a new `df_...` variable.

### Q8: Filter for rows where `age` is between 18 and 30 (inclusive) and `gender` is `'F'`.

(Mapped from `subscription == 'Premium'` → `gender == 'F'`, since this file has no `subscription` column.)

In [8]:
df_filtered = df.filter(col("age").between(18, 30) & (col("gender") == "F"))
df_filtered.select("transaction_id", "age", "gender").show(5)

+--------------+----+------+
|transaction_id| age|gender|
+--------------+----+------+
|     TXN152131|29.0|     F|
|     TXN147316|25.0|     F|
|     TXN102137|30.0|     F|
|     TXN102407|19.0|     F|
|     TXN107525|29.0|     F|
+--------------+----+------+
only showing top 5 rows


### Q9: Why is it often better to handle null values before aggregations like `sum()` or `avg()`?

Aggregate functions silently skip nulls, which shrinks the effective denominator and can quietly distort an average or sum without raising any error. Handling nulls first (drop or fill with an explicit default) makes that decision intentional rather than accidental, and keeps the result reproducible.

### Q10: Cast a string date column to `TimestampType` and rename it.

This is where the real messiness of `purchase_date` actually bites. The column mixes **three different date formats**: ISO (`2023-12-22`), day/month-with-slashes (`04/10/2022`), and month-day-with-dashes (`08-28-2021`). Casting with a single fixed format (`yyyy-MM-dd`) — exactly what the original synthetic exercise expected to "just work" — will correctly parse the ISO-style rows and turn every other format into `NULL` (using `try_to_timestamp` instead of a plain cast so non-matching rows don't crash the whole job).

In [9]:
df_timestamped = df.withColumn(
        "purchase_timestamp",
        try_to_timestamp(col("purchase_date"), lit("yyyy-MM-dd"))
    ).drop("purchase_date")

df_timestamped.select("transaction_id", "purchase_timestamp").show(5)

non_null_original_dates = df.filter(col("purchase_date").isNotNull()).count()
parsed_ok = df_timestamped.filter(col("purchase_timestamp").isNotNull()).count()
print(f"Rows with a non-null purchase_date string: {non_null_original_dates}")
print(f"Rows successfully parsed with format 'yyyy-MM-dd': {parsed_ok}")
print(f"Rows that FAILED to parse (different format in source): {non_null_original_dates - parsed_ok}")

+--------------+-------------------+
|transaction_id| purchase_timestamp|
+--------------+-------------------+
|     TXN130813|2023-12-22 00:00:00|
|     TXN152131|2023-06-07 00:00:00|
|     TXN117691|2023-05-28 00:00:00|
|     TXN157025|               NULL|
|     TXN117326|2023-07-17 00:00:00|
+--------------+-------------------+
only showing top 5 rows
Rows with a non-null purchase_date string: 57348
Rows successfully parsed with format 'yyyy-MM-dd': 34399
Rows that FAILED to parse (different format in source): 22949


### Q11: Explain the "Shuffle" process during a grouping operation. Why is it a wide transformation?

**The Shuffle:** redistributes data across partitions (and often across worker nodes) so all records sharing the same key end up together for aggregation — this is exactly what happened behind the scenes in the `groupBy` calls above (Q4, Q6, Q15).

**Wide transformation:** computing a single output partition requires reading and moving data from *multiple* input partitions over the network, making it far more resource-intensive than a narrow transformation like `filter()` or `select()`, which can be computed partition-by-partition with no network movement.

### Q12: Remove rows where `email` is null OR `customer_name` is empty/null.

(Mapped from `username` → `customer_name`, since this file has no `username` column — and `customer_name` has no empty strings, only nulls, so the check is `isNotNull()` on both.)

In [10]:
df_valid_users = df.filter(col("email").isNotNull() & col("customer_name").isNotNull())

print(f"Original row count: {df.count()}")
print(f"Rows with valid email/customer_name: {df_valid_users.count()}")
df_valid_users.select("transaction_id", "email", "customer_name").show(5)

Original row count: 62425
Rows with valid email/customer_name: 59192
+--------------+--------------------+-------------+
|transaction_id|               email|customer_name|
+--------------+--------------------+-------------+
|     TXN130813|linda.mehta925@gm...|  Linda Mehta|
|     TXN152131|vihaan.wilson243@...|Vihaan Wilson|
|     TXN157025|navya.singh685@ya...|  Navya Singh|
|     TXN117326|navya.kumar227@ho...|  Navya Kumar|
|     TXN135622|ishaan.mehta974@g...| Ishaan Mehta|
+--------------+--------------------+-------------+
only showing top 5 rows


### Q13: Use `agg()` to calculate min, max, and mean of `unit_price` in one call.

(Mapped from `price` → `unit_price`, since this file has no standalone `price` column.)

In [11]:
df_stats = df.agg(
    _min("unit_price").alias("min_unit_price"),
    _max("unit_price").alias("max_unit_price"),
    mean("unit_price").alias("mean_unit_price")
)

df_stats.show()

+--------------+--------------+-----------------+
|min_unit_price|max_unit_price|  mean_unit_price|
+--------------+--------------+-----------------+
|           2.0|        192.34|46.16459855769274|
+--------------+--------------+-----------------+



### Q14: Risk of `inferSchema=True` when source dates are messy/inconsistent.

This is no longer hypothetical — Q10 above demonstrates it directly. Because `purchase_date` mixes `YYYY-MM-DD`, `DD/MM/YYYY`, and `MM-DD-YYYY` styles in the *same column*, Spark's schema inference falls back to plain `StringType` rather than a native date type (which is exactly the dtype `printSchema()` showed at the top). That pushes the parsing problem downstream: a naive cast to a single format only recovers the rows matching that one format and quietly nulls out the rest — 22,949 rows in this file's case — unless you explicitly detect and handle each format (e.g. with `coalesce()` over several `try_to_timestamp()` calls, one per format) before treating the column as a real date.

### Q15: Final processing pipeline — drop duplicates, fill null amounts with 0, group to get total revenue.

(Mapped `store_id` → cleaned `region`, since this file has no `store_id` column. We also re-apply the `trim()`/`initcap()` normalization from Q4 — otherwise `region`'s casing/whitespace variants would split into ~26 separate, meaningless groups instead of 5 real regions.)

In [12]:
final_pipeline_df = df.dropDuplicates() \
                      .na.fill({"purchase_amount": 0.0}) \
                      .withColumn("region_clean", initcap(trim(col("region")))) \
                      .groupBy("region_clean") \
                      .agg(_sum("purchase_amount").alias("total_revenue"))

final_pipeline_df.orderBy("total_revenue", ascending=False).show()

+------------+------------------+
|region_clean|     total_revenue|
+------------+------------------+
|     Central| 2406300.150000006|
|        West|2403828.1100000036|
|        East|  2387193.46000001|
|       North|2365470.1900000037|
|       South| 2336714.119999996|
|        NULL|364047.60000000015|
+------------+------------------+



---
### Summary of what the cleaning revealed

- **2,400** exact duplicate rows (plus 1 fully-blank sentinel row) removed.
- **22,949** `purchase_date` values don't match a single fixed date format — a real schema-inference trap.
- **26** raw spellings of `region` collapse to 5 real regions once trimmed/case-normalized; the same applies to `category` (41 raw spellings → 9 real categories).
- Nulls are spread across nearly every column, most heavily in `rating` (4,480), `purchase_date` (5,077), and `signup_date` (4,881).
- Total revenue is fairly evenly split across regions (~$2.3–2.4M each) once `region` is cleaned, with ~$364K landing in an unclassified/NULL region bucket that would merit a closer look (likely tied to the same blank/null rows seen elsewhere).
